In [5]:
import sqlite3
import os 
import json 
from langchain_core.documents import Document
#from langchain_community.document_loaders import DatabaseLoader

In [14]:
base_path= os.path.abspath(os.path.join(os.getcwd() , '..'))
db_path=f"{base_path}/otherfile/databases"
os.makedirs(f"{db_path}",exist_ok=True)

In [20]:
conn=sqlite3.connect(f'{db_path}/vineet.db')
cursor = conn.cursor()


In [21]:
create_query ='create table if not exists orders( order_id int , order_date date, customer_id id , status string)'
cursor.execute(create_query)

In [24]:
insert_query = "insert into orders (order_id, order_date, customer_id , status) values( 1, '2026-05-26' , 111,'shipped' )"
cursor.execute(insert_query)

In [ ]:
conn.commit()
cursor.close()


In [ ]:
from typing import List 
from langchain_core.documents import Document
def sql_to_documents(db_path :str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents=[] 

    #case 1 
    cursor.execute('select * from sqlite3_master where type="table" ;')
    tables =cursor.fetchall()
    print(tables)

    for table in tables:
        table_name = table[0]
        cursor.execute(f"PRAGMA table_info {table_name}")
        columns = cursor.fetchall()

        column_names = [col[1] for col in columns]


        #Get table data
        cursor.execute(f'select * from {table_name}')
        rows= cursor.fetchall()  

        table_content =f"Sample Records \n"
        table_content += f"Column { ' ,'.join(column_names)}"  
        table_content += f"Total Records : {len(rows)}"

        for row in rows:
            record = dict(zip(column_names,row))
            table_content += f"{record}\n"

        doc = Document(page_content= table_content ,
                       metadata=  {
                           "source" : db_path,
                           "table_name" : table_name,
                           "num_records" : len(rows)

                       })   
        documents.add(doc)

    return documents




